[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/regge_calculus/regge_calculus.ipynb)

# Regge Calculus

This example computes the intrinsic metric, Gaussian curvature, and steady electrical conduction of a triangulated surface from its edge lengths alone, following Regge (1961). Within each triangle, three edge lengths fix every internal distance, area, and corner angle without placing the surface in ambient space. Around an interior vertex, corner angles that sum to less than a full turn curve the surface positively like a cap, while angles exceeding a full turn curve it negatively like a saddle. Carrying those prescribed lengths on a flat reference triangulation also determines how steady current bends across the curved surface between boundary electrodes.

The term ***duality map*** here means a linear map `Plane <- Point` from displacements to planes, and each triangle's intrinsic metric is modelled as a symmetric duality map built from products of corner slopes so that `metric(edge) & edge` recovers each prescribed squared length. The term ***simplex gradient*** here means each corner's linear slope `side / (2 * patch)` across the reference triangle, whose adjugate pulls local responses back to the vertices. Pairing the metric wedge with the reference patch `(metric(sides[0]) ^ metric(sides[1])) & patch` gives twice the squared intrinsic area, while combining side pairings with twice that area in the face's turning plane forms unit corner rotors whose logarithms sum to the vertex angle defects. Weighting side dyads `sides * (Plane & sides)` by the corner cotangents gives each triangle's area-weighted inverse metric, and `gradient.adjugate()(conductivities(gradient))` assembles steady conduction across the surface.

In classical Regge calculus and cotangent finite-element notation, a triangle with edge lengths $\ell_1, \ell_2, \ell_3$ and semiperimeter $s = (\ell_1 + \ell_2 + \ell_3)/2$ has area $A_T = \sqrt{s(s - \ell_1)(s - \ell_2)(s - \ell_3)}$ and corner angles $\theta_k = \arccos((\ell_i^2 + \ell_j^2 - \ell_k^2)/(2\ell_i\ell_j))$, vertex angle defects are scalar sums $\delta_v = 2\pi - \sum_{T \ni v} \theta_{T,v}$, and the intrinsic Laplace–Beltrami operator is assembled edge by edge from opposite-corner cotangents as $\frac{1}{2}\sum_{j \sim i}(\cot\alpha_{ij} + \cot\beta_{ij})(u_i - u_j)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA2D as ga
from numga.sparse import spdiag
from numga.topology import Topology
from examples.util import shapes
from examples.util.animation import save_figure
from examples.surfaces.regge_calculus import render, scenarios

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                  # Scalar+
Plane = ga.gatype.vector()                                                   # Vector+
Bivector = ga.gatype.bivector()                                              # Bivector+
Rotor = ga.gatype.rotor()                                                    # Rotor+
Point = ga.gatype.antivector()                                               # Antivector-
Antiscalar = ga.gatype.antiscalar()                                          # Antiscalar-
Metric = ga.gatype((Plane, Point))                                           # Plane+ <- Point-
Conductivity = ga.gatype((Point, Plane))                                     # Point- <- Plane+
Gradient = ga.gatype((Plane, Scalar))                                        # Plane+ <- Scalar+
Conductance = ga.gatype((Antiscalar, Scalar))                                # Antiscalar- <- Scalar+
triangle = Topology(3, [[0, 1, 2]])

np.set_printoptions(precision=4, suppress=True)

## Edge-length duality maps

Three prescribed squared edge lengths on a triangle fix a unique symmetric duality map from displacements to planes. Each of its three basis maps responds to one side and vanishes along the directions of the other two, giving each squared edge length an independent role in reconstructing the triangle's metric.

In [ ]:
n_triangle_subdivisions = 24
edge_stretches = np.array([1.25, 0.65, 1.45])                                # [3]

In [ ]:
def metric_basis(topology: Topology, vertices: Point) -> Metric:
    """Each triangle's three symmetric simplex-gradient products (`[3] (Plane+ <- Point-)[F]`) pair to one on their own side and zero on the other two."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)             # [3] Point-[F]
    patches = (sides[0] & sides[1]) / 2                                      # Patch-[F]
    gradients = sides / (2 * patches)                                        # [3] Plane+[F]
    # Along the two sides meeting at a corner, one endpoint gradient vanishes while on the opposite
    # side the two endpoint gradients pair to -1 and +1, making their symmetric product dual to the squared edge lengths.
    tails, heads = topology.corners[2, 1]
    first, second = gradients[tails], gradients[heads]                       # [3] Plane+[F], [3] Plane+[F]
    return -(first * (second & Point) + second * (first & Point)) / 2        # [3] (Plane+ <- Point-)[F]


corners, sample_points = scenarios.single_triangle(n_triangle_subdivisions)  # Point-[V], [...] Point-
triangle_basis = metric_basis(triangle, corners)                             # [3] (Plane+ <- Point-)[F]
triangle_squared_lengths = (
    (triangle.d[0] * corners).norm_squared() * mv.scalar(edge_stretches[:, None]).field()
)                                                                            # Scalar+[E]
triangle_metric = (spdiag(triangle_basis) * triangle.incidence[2, 1]).sum(axis=0) * triangle_squared_lengths  # (Plane+ <- Point-)[F]
save_figure(
    render.metric_basis_figure(corners, sample_points, triangle_basis, triangle_metric),
    "regge_metric_basis",
);

Each basis map reads its own edge in orange and vanishes along two dark-blue null lines parallel to the other two edges, changing sign across them. Weighted by the squared edge lengths, their sum in the fourth panel forms a positive-definite metric whose orange equal-length ellipses contrast with the dashed circles of the reference triangle.

In coordinate tensor notation with barycentric coordinates $\lambda_i$, each edge basis tensor reads $\Psi_{ij} = -\frac{1}{2}(d\lambda_i \otimes d\lambda_j + d\lambda_j \otimes d\lambda_i)$ and the piecewise-constant metric tensor is $g_T = \sum_{i < j} \ell_{ij}^2 \Psi_{ij}$.

The reconstructed metric also determines the triangle's intrinsic area and unit corner rotors, whose logarithms give the three corner turns.

In [ ]:
def intrinsic_areas(topology: Topology, vertices: Point, metrics: Metric) -> Scalar:
    """Each triangle's metric wedge paired with its reference patch determines its intrinsic area (`[...] Scalar+[F]`)."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)             # [3] Point-[F]
    patches = (sides[0] & sides[1]) / 2                                      # Patch-[F]
    # The wedge carries the duality map's area scaling; pairing it with the reference patch
    # and dividing by two gives the intrinsic area squared.
    return (((metrics(sides[0]) ^ metrics(sides[1])) & patches) / 2).square_root()  # [...] Scalar+[F]


def corner_rotors(topology: Topology, vertices: Point, metrics: Metric, areas: Scalar) -> Rotor:
    """Side pairings and twice the intrinsic area in each face's turning plane form unit corner rotors (`[..., 3] Rotor+[F]`)."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)             # [3] Point-[F]
    patches = (sides[0] & sides[1]) / 2                                      # Patch-[F]
    turning_planes = patches.normalized().dual_inverse()                     # Bivector+[F]
    tails, heads = topology.corners[2, 1]
    # Combining the metric pairing of the two sides meeting at each corner with twice the intrinsic
    # area in the face's turning plane forms a rotor whose norm is the product of those two side lengths.
    dots = -(metrics[..., None](sides[tails]) & sides[heads])                # [..., 3] Scalar+[F]
    return (dots + 2 * areas[..., None] * turning_planes).normalized()       # [..., 3] Rotor+[F]


triangle_sides = triangle.boundary[2, 1] * (triangle.d[0] * corners)         # [3] Point-[F]
reference_area = ((triangle_sides[0] & triangle_sides[1]) / 2).antinorm()    # Scalar+[F]
triangle_area = intrinsic_areas(triangle, corners, triangle_metric)          # Scalar+[F]
triangle_rotors = corner_rotors(triangle, corners, triangle_metric, triangle_area)  # [3] Rotor+[F]
triangle_turns = triangle_rotors.log()                                       # [3] Bivector+[F]

In [ ]:
print("Reference vs intrinsic area:", reference_area.batch()[0].to_array(), triangle_area.batch()[0].to_array())
print("Corner turns (radians):", (triangle_turns.batch()[:, 0] & mv.xy.dual()).to_array())

## Curvature from corner turns

One flat triangulation of a disk can carry distinct families of intrinsic edge lengths. Lengthening radial steps while shortening circumferential ones leaves less than a full turn around each interior vertex, whereas lengthening circumferential steps leaves an excess turn. Dividing each vertex's angle defect by its dual-cell share of the surrounding intrinsic triangle areas gives the Gaussian curvature bivector on the fixed reference disk.

In [ ]:
n_disk_divisions = 9
disk_radius = 1.0
radial_rates = np.array([0.0, 3.2, -0.45])                                   # [3]
hoop_rates = np.array([0.0, -0.78, 3.2])                                     # [3]

In [ ]:
def vertex_defects(topology: Topology, vertices: Point, corner_turns: Bivector) -> Bivector:
    """Interior vertices fall short of a full turn and boundary vertices of a half turn by the angle defect (`[...] Bivector+[V]`)."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)             # [3] Point-[F]
    turning_planes = ((sides[0] & sides[1]) / 2).normalized().dual_inverse()  # Bivector+[F]
    corner_mean = topology.incidence[2, 0].mean(axis=0)                      # Scalar+[F <- V]
    full_turns = 2 * np.pi * (corner_mean.adjugate() * turning_planes).normalized()  # Bivector+[V]
    # A flat surface turns through a full turn around an interior vertex, and through half of one at the rim.
    rim = topology.reading(topology.exterior[0], 0)                          # Scalar+[V_rim <- V]
    target_turns = full_turns - rim.adjugate() * (rim * full_turns) / 2      # Bivector+[V]
    return target_turns - (topology.incidence[2, 0].adjugate() * corner_turns).sum(axis=-1)  # [...] Bivector+[V]


topology, coordinates = shapes.hexagonal_disk(disk_radius, n_disk_divisions)
vertices = mv.vector(coordinates).dual().field()                             # Point-[V]
edges = topology.d[0] * vertices                                             # Point-[E]
midpoints = topology.incidence[1, 0].mean(axis=0) * vertices                 # Point-[E]
# The inner product measures radial motion times radius; the join measures swept area,
# hence circumferential motion times radius. Their squares choose the two length responses.
edge_squared_lengths = (
    edges.norm_squared()
    + radial_rates * (midpoints | edges).squared()
    + hoop_rates * (midpoints & edges).norm_squared()
)                                                                            # [3] Scalar+[E]

disk_basis = metric_basis(topology, vertices)                                # [3] (Plane+ <- Point-)[F]
metrics = (spdiag(disk_basis) * topology.incidence[2, 1]).sum(axis=0) * edge_squared_lengths  # [3] (Plane+ <- Point-)[F]
areas = intrinsic_areas(topology, vertices, metrics)                         # [3] Scalar+[F]
turns = corner_rotors(topology, vertices, metrics, areas).log()              # [3, 3] Bivector+[F]
defects = vertex_defects(topology, vertices, turns)                          # [3] Bivector+[V]
dual_areas = topology.incidence[2, 0].mean(axis=0).adjugate() * areas        # [3] Scalar+[V]
curvatures = defects / dual_areas                                            # [3] Bivector+[V]
centres = topology.incidence[2, 0].mean(axis=0) * vertices                   # Point-[F]

save_figure(
    render.curvature_defects_figure(topology, vertices, centres, metrics, curvatures),
    "regge_curvature_defects",
);

The flat left disk has zero interior curvature, radial stretching curves the middle disk positively in orange, and circumferential stretching curves the right disk negatively in blue. Each sampled indicatrix ellipse encloses displacements of equal intrinsic length against a dashed reference circle, coloured red where radial stretch dominates and blue where circumferential stretch dominates.

In scalar angle-defect notation, interior defects $\delta_v = 2\pi - \sum_{T \ni v}\theta_{T,v}$ and boundary geodesic turns $\delta_v^\partial = \pi - \sum_{T \ni v}\theta_{T,v}$ satisfy the discrete Gauss–Bonnet identity $\sum_{v \notin \partial M}\delta_v + \sum_{v \in \partial M}\delta_v^\partial = 2\pi\chi(M)$.

In [ ]:
print("Gaussian curvature at centre (radial stretch, hoop stretch):", (curvatures.batch()[1:, 0] & mv.xy.dual()).to_array())

## Steady conduction on the intrinsic surface

A potential slope across each triangle drives current through that triangle's conductivity map. Even when the material conducts isotropically in its own intrinsic metric, curved edge lengths redirect the flow when viewed on the flat reference disk. Pulling every triangle's current back through the simplex gradient's adjugate balances flow at the interior vertices between two boundary electrodes.

In [ ]:
electrode_threshold = 0.78

In [ ]:
def triangle_conductivities(topology: Topology, vertices: Point, metrics: Metric, areas: Scalar) -> Conductivity:
    """Corner cotangents weight the side dyads into each triangle's area-weighted inverse metric (`[...] (Point- <- Plane+)[F]`)."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)             # [3] Point-[F]
    tails, heads = topology.corners[2, 1]
    dots = -(metrics[..., None](sides[tails]) & sides[heads])                # [..., 3] Scalar+[F]
    # Each side dyad turns a potential slope into flow along that side;
    # the corner cotangents weight the isotropic response of the intrinsic triangle.
    cotangents = dots / (2 * areas[..., None])                               # [..., 3] Scalar+[F]
    return ((cotangents / 2) * (sides * (Plane & sides))).sum(axis=-1)       # [...] (Point- <- Plane+)[F]


conductivities = triangle_conductivities(topology, vertices, metrics, areas)  # [3] (Point- <- Plane+)[F]

In [ ]:
def simplex_gradient(topology: Topology, vertices: Point) -> Gradient:
    """Corner simplex gradients assemble into the piecewise-linear gradient operator (`(Plane+ <- Scalar+)[F <- V]`) on the reference triangulation."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)             # [3] Point-[F]
    patches = (sides[0] & sides[1]) / 2                                      # Patch-[F]
    gradients = sides / (2 * patches)                                        # [3] Plane+[F]
    return (spdiag(gradients) * topology.incidence[2, 0]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[F <- V]


def steady_potential(topology: Topology, conductance: Conductance,
                     free_indices: np.ndarray, boundary_potential: Scalar) -> Scalar:
    """Current balance at every unpinned vertex determines the steady potential (`[...] Scalar+[V]`) from `boundary_potential`."""
    free = topology.reading(free_indices, 0).adjugate() * Scalar             # (Scalar+ <- Scalar+)[V <- V_free]
    rhs = -free.adjugate()(conductance(boundary_potential))                  # [...] Antiscalar-[V_free]
    return boundary_potential + free(free.adjugate()(conductance(free)).solve(rhs))  # [...] Scalar+[V]


V, E, F = topology.n_elements
rim_indices = topology.exterior[0]
rim = topology.reading(rim_indices, 0)                                       # Scalar+[V_rim <- V]
electrodes = rim_indices[(mv.x & (rim * vertices)).abs() > electrode_threshold]
electrode_reading = topology.reading(electrodes, 0)                          # Scalar+[V_electrodes <- V]
boundary_potential = electrode_reading.adjugate() * (electrode_reading * (mv.x & vertices))  # Scalar+[V]
free_indices = np.setdiff1d(np.arange(V), electrodes)

gradient = simplex_gradient(topology, vertices)                              # (Plane+ <- Scalar+)[F <- V]
# The adjugate carries each triangle's current response back to its vertex potentials.
conductance = gradient.adjugate()(conductivities(gradient))                  # [3] (Antiscalar- <- Scalar+)[V <- V]
potential = steady_potential(topology, conductance, free_indices, boundary_potential)  # [3] Scalar+[V]
reference_sides = topology.boundary[2, 1] * edges                            # [3] Point-[F]
reference_areas = ((reference_sides[0] & reference_sides[1]) / 2).antinorm()  # Scalar+[F]
current = -conductivities(gradient(potential)) / reference_areas             # [3] Point-[F]

save_figure(
    render.intrinsic_flow_figure(topology, vertices, potential, current),
    "regge_intrinsic_laplacian",
);

Shorter circumferential distances in the middle disk draw current toward the rim, whereas longer circumferential distances in the right disk focus current through the centre. Background colour and level contours show the vertex potential between the right and left electrodes. Dark streaks taper downstream with length proportional to the reference-frame current speed on one scale shared across all three panels.

In coordinate tensor and finite-element notation, each triangle's area-weighted inverse metric reads $A_T g_T^{-1} = \frac{1}{2}\sum_{k=1}^3 \cot\theta_k\, s_k \otimes s_k$, and pulling it back through the piecewise-linear basis gradients $\nabla\lambda_i$ yields the intrinsic cotangent stiffness matrix.

In [ ]:
origin_face = centres.antinorm().argmin()
print("Reference current magnitude at centre (flat, radial stretch, hoop stretch):", current.batch()[:, origin_face].antinorm().to_array())

In [ ]:
# checks
check_sides = triangle_sides.batch()[:, 0]
side_readings = triangle_basis.batch()[:, 0, None](check_sides[None, :]) & check_sides[None, :]
assert ((side_readings - np.eye(3)).abs() < 1e-10).all()

prescribed_side_lengths = (triangle.incidence[2, 1] * triangle_squared_lengths).batch()[:, 0]
assert (((triangle_metric.batch()[0](check_sides) & check_sides) - prescribed_side_lengths).abs() < 1e-10).all()

side_lengths = prescribed_side_lengths.square_root()
semiperimeter = side_lengths.sum() / 2
heron_area = (
    semiperimeter
    * (semiperimeter - side_lengths[0])
    * (semiperimeter - side_lengths[1])
    * (semiperimeter - side_lengths[2])
).square_root()
assert ((triangle_area.batch()[0] - heron_area).abs() < 1e-10).all()
assert ((triangle_rotors.batch()[:, 0].scalar_norm_squared() - 1).abs() < 1e-10).all()
assert ((triangle_turns.batch()[:, 0].sum(axis=0) - np.pi * mv.xy).norm() < 1e-10).all()
assert ((triangle_rotors.batch()[0, 0] * triangle_rotors.batch()[1, 0] * triangle_rotors.batch()[2, 0] + 1).norm() < 1e-10).all()

assert ((defects.sites.sum() - 2 * np.pi * mv.xy).norm() < 1e-10).all()
assert ((topology.reading(topology.interior[0], 0) * defects[0]).batch().norm() < 1e-10).all()
assert (curvatures.batch()[1, 0] & mv.xy.dual()) > 0
assert (curvatures.batch()[2, 0] & mv.xy.dual()) < 0

assert (np.abs((metrics.batch()(conductivities.batch()) - areas.batch() * Plane).kernel) < 1e-10).all()
reciprocity = (
    (potential & conductance(boundary_potential)).sites.sum()
    - (boundary_potential & conductance(potential)).sites.sum()
)
assert (reciprocity.abs() < 1e-10).all()
assert ((topology.reading(free_indices, 0) * conductance(potential)).batch().norm() < 1e-9).all()